# **Modelo de Recomendação INERI:**

Este notebook implementa um sistema de recomendação utilizando a técnica de filtragem colaborativa baseada em fatoração de matriz, por meio do algoritmo SVD (Singular Value Decomposition), disponibilizado pela biblioteca Surprise. A filtragem colaborativa é um método que recomenda itens a partir de padrões de comportamento observados em diferentes usuários, partindo do princípio de que usuários que demonstraram preferências semelhantes tendem a manter padrões próximos em suas escolhas futuras.

No contexto deste projeto, cada conta é tratada como um usuário, enquanto cada ativo disponível é considerado um item. As avaliações (ratings) utilizadas pelo modelo são obtidas a partir de uma normalização percentual da alocação financeira de cada ativo dentro da carteira do cliente, permitindo transformar dados financeiros em um formato compatível com técnicas de recomendação.

Embora o conceito inicial de filtragem colaborativa possa ser dividido entre abordagens baseadas em usuário (user-based) ou em item (item-based), o algoritmo SVD pertence à categoria model-based, que trabalha de forma indireta. Em vez de comparar diretamente usuários entre si ou itens entre si, o SVD fatora a matriz de interações entre usuários e itens em representações vetoriais de dimensões latentes, permitindo prever a nota que um usuário daria a um item com base em padrões implícitos da base de dados.

Dessa forma, este projeto utiliza uma abordagem model-based, que considera simultaneamente padrões de comportamento de usuários e itens ao realizar suas previsões de notas e recomendações.

**(Como decidido na AAD, mativemos a database da forma como ela foi enviada, realizando uma EDA, mas sem alterações.)**

---

## **Tópicos**

A seguir, serão apresentadas as seções de **1.0 a 3.0**, que descrevem as etapas para a construção do modelo:

* **1.0 — Identificação de clientes desenquadrados:**
  Nesta etapa, classificamos aproximadamente 80 tipos de ativos distintos e calculamos a proporção de cada um em relação ao investimento total de cada cliente. Em seguida, comparamos essas proporções com os percentuais ideais estabelecidos, identificando assim as carteiras desenquadradas.

* **2.0 — Recomendação de ativos:**
  Desenvolvemos um modelo híbrido, utilizando o algoritmo **SVD** da biblioteca **Surprise**, para recomendar 5 ativos diferentes a cada cliente com carteira desenquadrada. As sugestões são baseadas no comportamento de investimento de outros clientes com perfis semelhantes.

* **3.0 — Análise de resultados:**
  Por fim, avaliamos o desempenho do modelo por meio de uma análise gráfica dos dados e das métricas **RMSE** e **MAE**, validando a precisão e a eficácia das recomendações geradas.


---
# 1.0 Definição de perfis desenquadrados

Esta seção tem como objetivo listar os perfis quem apresentam diferenaça entre perfil de investidor e investimentos reais.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!ls "/content/drive/My Drive/modelog1"

In [ ]:
!pip install numpy==1.23.5 #instalação da versão do numpy que não dá conflito com o Surprise
!pip install scikit-surprise
import pandas as pd
import os
import numpy as np
from surprise import Dataset, Reader, SVD
import pickle

### **Importação dos datasets**

In [ ]:
carteiras = pd.read_csv("/content/drive/My Drive/modelog1/carteiras.csv")
ativos = pd.read_csv("/content/drive/My Drive/modelog1/ativos.csv")
carteiras.columns = ['Conta', 'Financeiro', 'Quantidade', 'Nome Ativo', 'Tipo de Ativo', 'Perfil da carteira']
ativos.columns = ['nome_ativo', 'tipo']
ativos_classificados = pd.read_csv("/content/drive/My Drive/modelog1/ativos_classificados.csv")
carteiras.columns = ['Conta', 'Financeiro', 'Quantidade', 'Nome Ativo', 'Tipo de Ativo', 'Perfil da carteira']
ativos_classificados.columns = ['Tipo de Ativo', 'Nivel de Risco']

print(ativos.head())
print(carteiras.head())

### **Mapeamento de perfil baseado no Nome do Ativo**

In [ ]:
# Função para mapear a classificação da base ativos.csv
def mapear_classificacao(tipo):
    if tipo == 'Arrojado':
        return 'Renda Variável'
    elif tipo == 'Sofisticado':
        return 'Outro'
    elif tipo == 'Conservador':
        return 'Renda Fixa'
    else:
        return 'Outro'  # Para qualquer outro caso, como segurança

# Fazer o merge entre carteiras e ativos e mapear os tipos
df = carteiras.merge(ativos, left_on='Nome Ativo', right_on='nome_ativo', how='left')
df['tipo_mapeado'] = df['tipo'].apply(mapear_classificacao)

# Função para calcular proporções por carteira
def calcular_proporcoes(df):
    # Agrupar por carteira (Conta) e tipo mapeado, somando os valores financeiros
    agrupado = df.groupby(['Conta', 'Perfil da carteira', 'tipo_mapeado'])['Financeiro'].sum().unstack(fill_value=0)

    # Calcular o total por carteira
    agrupado['total'] = agrupado.sum(axis=1)

    # Calcular proporções
    tipos_ativos = ['Renda Fixa', 'Renda Variável', 'Outro']
    for tipo in tipos_ativos:
        if tipo not in agrupado.columns:
            agrupado[tipo] = 0
        agrupado[f'prop_{tipo}'] = agrupado[tipo] / agrupado['total']

    return agrupado.reset_index()

# Função para verificar enquadramento
def verificar_enquadramento(proporcoes):
    resultados = []

    for _, row in proporcoes.iterrows():
        perfil = row['Perfil da carteira']
        prop_rf = row['prop_Renda Fixa'] if 'prop_Renda Fixa' in row else 0
        prop_rv = row['prop_Renda Variável'] if 'prop_Renda Variável' in row else 0
        prop_outro = row['prop_Outro'] if 'prop_Outro' in row else 0

        enquadrado = False

        if perfil == 'Moderado':
            # 60% Renda Fixa, 40% demais
            enquadrado = (prop_rf >= 0.60) and (prop_rv + prop_outro <= 0.40)

        resultados.append({
            'conta': row['Conta'],
            'perfil': perfil,
            'prop_Renda Fixa': prop_rf,
            'prop_Renda Variável': prop_rv,
            'prop_Outro': prop_outro,
            'enquadrado': 'Sim' if enquadrado else 'Não'
        })

    return pd.DataFrame(resultados)

# Função principal
def main():
    # Calcular proporções
    proporcoes = calcular_proporcoes(df)

    # Verificar enquadramento
    resultado = verificar_enquadramento(proporcoes)

    # Exibir ou salvar resultado
    print(resultado)
    resultado.to_csv('resultado_enquadramento_Nome.csv', index=False)

# Executar
if __name__ == "__main__":
    main()

### **Mapeamento de perfil baseado no Tipo do Ativo**

In [ ]:
# Função para mapear o Nivel de Risco para categorias finais
def mapear_classificacao(nivel_risco):
    if nivel_risco == 'Renda Fixa':
        return 'Renda Fixa'
    elif nivel_risco == 'Renda Variável':
        return 'Renda Variável'
    elif nivel_risco == 'Outros':
        return 'Outro'
    else:
        return 'Outro'  # Para segurança

# Fazer o merge entre carteiras e ativos_classificados
df = carteiras.merge(ativos_classificados, on='Tipo de Ativo', how='left')
df['categoria'] = df['Nivel de Risco'].apply(mapear_classificacao)

# Função para calcular proporções por carteira
def calcular_proporcoes(df):
    # Agrupar por carteira (Conta) e categoria, somando os valores financeiros
    agrupado = df.groupby(['Conta', 'Perfil da carteira', 'categoria'])['Financeiro'].sum().unstack(fill_value=0)

    # Calcular o total por carteira
    agrupado['total'] = agrupado.sum(axis=1)

    # Calcular proporções
    tipos_ativos = ['Renda Fixa', 'Renda Variável', 'Outro']
    for tipo in tipos_ativos:
        if tipo not in agrupado.columns:
            agrupado[tipo] = 0
        agrupado[f'prop_{tipo}'] = agrupado[tipo] / agrupado['total']

    return agrupado.reset_index()

# Função para verificar enquadramento
def verificar_enquadramento(proporcoes):
    resultados = []

    for _, row in proporcoes.iterrows():
        perfil = row['Perfil da carteira']
        prop_rf = row['prop_Renda Fixa'] if 'prop_Renda Fixa' in row else 0
        prop_rv = row['prop_Renda Variável'] if 'prop_Renda Variável' in row else 0
        prop_outro = row['prop_Outro'] if 'prop_Outro' in row else 0

        enquadrado = False

        if perfil == 'Conservador':
            # 90% Renda Fixa, 10% demais
            enquadrado = (prop_rf >= 0.90) and (prop_rv + prop_outro <= 0.10)
        elif perfil == 'Moderado':
            # 60% Renda Fixa, 40% demais
            enquadrado = (prop_rf >= 0.60 and prop_rf < 0.90) and (prop_rv + prop_outro <= 0.40)
        elif perfil in ['Sofisticado', 'Arrojado']:
            # 70% Renda Variável, 30% demais
            enquadrado = (prop_rv >= 0.70) and (prop_rf + prop_outro <= 0.30)

        resultados.append({
            'conta': row['Conta'],
            'perfil': perfil,
            'prop_Renda Fixa': prop_rf,
            'prop_Renda Variável': prop_rv,
            'prop_Outro': prop_outro,
            'enquadrado': 'Sim' if enquadrado else 'Não'
        })

    return pd.DataFrame(resultados)

# Função principal
def main():
    # Calcular proporções
    proporcoes = calcular_proporcoes(df)

    # Verificar enquadramento
    resultado = verificar_enquadramento(proporcoes)

    # Exibir ou salvar resultado
    print(resultado)
    resultado.to_csv('resultado_enquadramento.csv', index=False)

# Executar
if __name__ == "__main__":
    main()

### **Comparar Classificação com base nas duas colunas**

A análise da proporção com base no Tipo de Ativo apresentou resultados mais realistas, isso pode ser explicado pela maior dificuldade de classificar cerca de 8102 ativos individualmente, aumentando a margem para erro e afetando a classificação.

In [87]:
enquadramentoTipo = pd.read_csv("resultado_enquadramento.csv")

print(enquadramentoTipo['enquadrado'].value_counts())

enquadramentoNome = pd.read_csv("resultado_enquadramento_Nome.csv")

print(enquadramentoNome['enquadrado'].value_counts())

enquadrado
Não    4096
Sim    2006
Name: count, dtype: int64
enquadrado
Não    5745
Sim     357
Name: count, dtype: int64


---
# 2.0 Criação do Modelo de Recomendação

Esta seção do notebook realiza tratará da criação do modelo de recomendação, optamos por utilizar a algoritmo de desenquadramento baseado no Tipo de Ativo, por mostrar maior precisão.




## 1️⃣ Carregando os dados
 Dados das carteiras, classificação dos ativos e informações de enquadramento de clientes.

In [62]:
df_enquadramento = pd.read_csv("/content/drive/My Drive/modelog1/resultado_enquadramento.csv")
df_carteiras = pd.read_csv("/content/drive/My Drive/modelog1/carteiras.csv")
df_classificacao = pd.read_csv("/content/drive/My Drive/modelog1/ativos_classificados.csv")

# 2️⃣ Renomeando colunas e ajustando dados
 Ajustamos os nomes das colunas para facilitar o merge e removemos espaços desnecessários.


In [63]:
df_carteiras.columns = ['Conta', 'Financeiro', 'Quantidade', 'Nome Ativo', 'tipo_ativo', 'Perfil da carteira']
df_classificacao.columns = ['tipo_ativo', 'Nivel de Risco']
df_classificacao['tipo_ativo'] = df_classificacao['tipo_ativo'].str.strip()

# 3️⃣ Mesclando informações de risco na carteira
 Juntamos o dataframe de carteiras com o de classificação de ativos para trazer o nível de risco.


In [64]:
df_carteiras = df_carteiras.merge(df_classificacao, on='tipo_ativo', how='left')

# 4️⃣ Verificando resultado do merge
 Validamos as colunas resultantes e os valores nulos.


In [ ]:
print(f"Colunas após merge: {df_carteiras.columns.tolist()}")
print(f"Valores nulos em 'Nivel de Risco': {df_carteiras['Nivel de Risco'].isna().sum()}")

# 5️⃣ Tratando valores nulos
 Substituímos riscos não identificados por 'Desconhecido'.

In [66]:
df_carteiras['Nivel de Risco'] = df_carteiras['Nivel de Risco'].fillna('Desconhecido')

# 6️⃣ Separando contas enquadradas e desenquadradas
 Criamos duas listas: contas enquadradas e contas que precisam de ajustes.


In [67]:
contas_enquadradas = df_enquadramento[df_enquadramento['enquadrado'] == 'Sim']['conta']
contas_desenquadradas = df_enquadramento[df_enquadramento['enquadrado'] == 'Não']['conta']

# 7️⃣ Criando rating proporcional para carteiras enquadradas
 Calculamos uma nota normalizada para o financeiro de cada ativo na carteira.


In [68]:
df_treino = df_carteiras[df_carteiras['Conta'].isin(contas_enquadradas)].copy()
df_treino['rating'] = df_treino.groupby('Conta')['Financeiro'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-10)
)

# 8️⃣ Treinamento do modelo de recomendação com Surprise (SVD)
 Montamos o dataset, preparamos o trainset e treinamos o modelo.


In [69]:
reader = Reader(rating_scale=(0, 1))
data = Dataset.load_from_df(df_treino[['Conta', 'Nome Ativo', 'rating']], reader)
trainset = data.build_full_trainset()
model = SVD()
model.fit(trainset)

# 9️⃣ Preparando dicionário de ativos únicos e lookup de nível de risco
 Evita repetições e agiliza buscas.


In [70]:
ativos_unicos = df_carteiras[['Nome Ativo', 'tipo_ativo', 'Nivel de Risco']].drop_duplicates()
dict_nivel_risco = dict(zip(ativos_unicos['Nome Ativo'], ativos_unicos['Nivel de Risco']))

# 🔟 Função para definir a prioridade de rebalanceamento
 Define qual tipo de ativo deve ser priorizado com base no perfil.


In [71]:
def calcular_necessidade_rebalanceamento(conta):
    perfil = df_enquadramento[df_enquadramento['conta'] == conta]['perfil'].values[0]
    if perfil == 'Conservador':
        return perfil, 'Renda Fixa'
    elif perfil == 'Moderado':
        return perfil, 'Renda Fixa'
    else:
        return perfil, 'Renda Variável'

# 🔄 Gerando recomendações para cada conta desenquadrada
 Para cada conta:
 - Obtém perfil e ativos já presentes.
 - Calcula necessidade de rebalanceamento.
 - Filtra ativos válidos.
 - Faz previsões com o modelo.
 - Ordena e seleciona os 5 melhores.
 - Registra o impacto na carteira.


In [72]:
recomendacoes = []

for conta in contas_desenquadradas:
    try:
        ativos_ja_tem = df_carteiras[df_carteiras['Conta'] == conta]['Nome Ativo'].unique()
        perfil_cliente, tipo_prioritario = calcular_necessidade_rebalanceamento(conta)
        if perfil_cliente == 'Conservador':
            tipos_aceitos = ['Renda Fixa']
        elif perfil_cliente == 'Moderado':
            tipos_aceitos = ['Renda Fixa', 'Outros']
        else:
            tipos_aceitos = ['Renda Fixa', 'Renda Variável', 'Outros']
        mascara = (~ativos_unicos['Nome Ativo'].isin(ativos_ja_tem)) & (ativos_unicos['Nivel de Risco'].isin(tipos_aceitos + ['Desconhecido']))
        ativos_validos_df = ativos_unicos[mascara]
        ativos_validos = ativos_validos_df['Nome Ativo'].values
        predicoes = []
        for ativo in ativos_validos:
            prediction = model.predict(conta, ativo).est
            nivel_risco = dict_nivel_risco.get(ativo, 'Desconhecido')
            bonus = 0.2 if nivel_risco == tipo_prioritario else 0
            score_final = prediction + bonus
            predicoes.append((ativo, score_final, nivel_risco))
        top_predicoes = sorted(predicoes, key=lambda x: x[1], reverse=True)[:5]
        for ativo, score, nivel_risco in top_predicoes:
            if nivel_risco == tipo_prioritario:
                impacto = f"Recomendado para rebalanceamento: aumenta {nivel_risco}"
            else:
                impacto = f"Compatível com o perfil {perfil_cliente}"
            recomendacoes.append((conta, ativo, score, nivel_risco, impacto))
    except Exception as e:
        print(f"Erro ao processar conta {conta}: {e}")
        continue


# 💾 Salvando as recomendações em CSV


In [73]:
df_recs = pd.DataFrame(recomendacoes,
                      columns=['Conta', 'Nome Ativo', 'Score', 'Nivel de Risco', 'Impacto'])
df_recs.to_csv("recomendacoes_por_conta.csv", index=False)

# 📊 Resumo das recomendações geradas


In [ ]:
print(f"Total de contas desenquadradas: {len(contas_desenquadradas)}")
print(f"Total de recomendações geradas: {len(df_recs)}")
print(f"Média de recomendações por conta: {len(df_recs)/max(1, len(contas_desenquadradas)):.2f}")
print("\nExemplo de recomendações:")
print(df_recs.head(10))

# 📦 Salvando o modelo treinado em arquivo pickle

In [ ]:
with open('modelo_recomendacao_final.pkl', 'wb') as arquivo_modelo:
    pickle.dump(model, arquivo_modelo)
print("Modelo salvo com sucesso!")

# 🔍 Função de inferência — prever score para um cliente e ativo específico


In [76]:
with open('modelo_recomendacao_final.pkl', 'rb') as arquivo_modelo:
    modelo_carregado = pickle.load(arquivo_modelo)

def recomendar_ativos_para_novo_usuario(ativos_ja_tem, modelo, ativos_disponiveis, n_recomendacoes=5):
    """
    Gera recomendações de ativos para uma nova conta, considerando os ativos que ela já possui.

    Parâmetros:
    - ativos_ja_tem: array ou lista com os nomes dos ativos que a conta já possui
    - modelo: modelo treinado do Surprise
    - ativos_disponiveis: DataFrame com ativos disponíveis e suas informações
    - n_recomendacoes: número de ativos recomendados a retornar (default=5)

    Retorna:
    - Lista de tuplas (Nome Ativo, Score previsto)
    """

    # Filtra ativos válidos (que a conta ainda não possui)
    ativos_validos = ativos_disponiveis[~ativos_disponiveis['Nome Ativo'].isin(ativos_ja_tem)]['Nome Ativo'].values

    predicoes = []

    # Faz predições de score para os ativos válidos
    for ativo in ativos_validos:
        predicao = modelo.predict("nova_conta", ativo)
        score_estimado = predicao.est
        predicoes.append((ativo, score_estimado))

    # Ordena e retorna os top N recomendados
    recomendacoes = sorted(predicoes, key=lambda x: x[1], reverse=True)[:n_recomendacoes]

    return recomendacoes


# 📝 Exemplo de uso da função de previsão de score


In [77]:
# Mock dos ativos já possuídos
ativos_ja_tem = ativos_unicos['Nome Ativo'].sample(3, random_state=1).values

# Gerar recomendações
recs_nova_conta = recomendar_ativos_para_novo_usuario(
    ativos_ja_tem=ativos_ja_tem,
    modelo=modelo_carregado,
    ativos_disponiveis=ativos_unicos,
    n_recomendacoes=5
)

# Exibir resultado
for ativo, score in recs_nova_conta:
    print(f"Recomendado: {ativo} | Score previsto: {score:.4f}")

Recomendado: LCI PRE | Score previsto: 0.6206
Recomendado: CDCA CDIE | Score previsto: 0.5855
Recomendado: LCI CDIE | Score previsto: 0.5186
Recomendado: POTE12 IPCA | Score previsto: 0.5128
Recomendado: MSGT23 IPCA | Score previsto: 0.4937


Simulamos um cenário onde um novo cliente possui determinados ativos em sua carteira. Para isso, selecionamos aleatoriamente três ativos reais disponíveis na base de dados. Como o modelo foi treinado apenas com dados das contas existentes, ele não conhece o histórico dessa nova conta e, por padrão, atribui uma média baseada no comportamento geral observado no treinamento. Assim, as recomendações são feitas considerando a similaridade desses ativos com o padrão aprendido no modelo, desconsiderando histórico individual prévio, mas respeitando as restrições de não recomendar ativos que o cliente já possui.

---
# 3.0 Análise de Resultados das Recomendações

Este notebook realiza a análise dos resultados de um sistema de recomendação de ativos financeiros. As análises envolvem:
- Cobertura de recomendações
- Qualidade do rebalanceamento
- Alinhamento entre perfil do cliente e recomendação
- Diversidade das recomendações
- Análise dos ativos mais recomendados
- Visualizações gráficas
- Métricas específicas por perfil de cliente




In [78]:
# 📦 Importação das bibliotecas necessárias
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from surprise import accuracy
from collections import Counter


## 1️⃣ Análise da Cobertura de Recomendações

Verifica quantas contas desenquadradas receberam recomendações e calcula a cobertura percentual.


In [79]:
recomendacoes_df = pd.read_csv("recomendacoes_por_conta.csv")
contas_com_recomendacao = recomendacoes_df['Conta'].nunique()
contas_desenquadradas_total = len(contas_desenquadradas)

print(f"=== ANÁLISE DE COBERTURA ===")
print(f"Total de contas desenquadradas: {contas_desenquadradas_total}")
print(f"Contas que receberam recomendações: {contas_com_recomendacao}")
print(f"Cobertura: {contas_com_recomendacao/contas_desenquadradas_total:.2%}")


=== ANÁLISE DE COBERTURA ===
Total de contas desenquadradas: 4096
Contas que receberam recomendações: 4096
Cobertura: 100.00%


## 2️⃣ Análise da Qualidade das Recomendações para Rebalanceamento

Avalia a distribuição dos níveis de risco das recomendações feitas.


In [80]:
print("\n=== ANÁLISE DE QUALIDADE DO REBALANCEAMENTO ===")
distribuicao_nivel_risco = recomendacoes_df['Nivel de Risco'].value_counts(normalize=True)
print("Distribuição das recomendações por nível de risco:")
print(distribuicao_nivel_risco)



=== ANÁLISE DE QUALIDADE DO REBALANCEAMENTO ===
Distribuição das recomendações por nível de risco:
Nivel de Risco
Renda Fixa        0.683398
Renda Variável    0.316602
Name: proportion, dtype: float64


## 3️⃣ Alinhamento entre Perfil e Recomendações

Verifica se as recomendações estão compatíveis com o perfil de risco do cliente.

A seguir:
- Junta o dataframe de recomendações com o perfil do cliente.
- Define uma função de avaliação de alinhamento.
- Aplica a função e conta a distribuição das classificações de alinhamento.


In [81]:
print("\n=== ALINHAMENTO PERFIL-RECOMENDAÇÃO ===")
recomendacoes_com_perfil = recomendacoes_df.merge(
    df_enquadramento[['conta', 'perfil']],
    left_on='Conta',
    right_on='conta',
    how='left'
)

def avaliar_alinhamento(row):
    perfil = row['perfil']
    nivel_risco = row['Nivel de Risco']

    if perfil == 'Conservador' and nivel_risco == 'Renda Fixa':
        return 'Ótimo'
    elif perfil == 'Moderado' and nivel_risco in ['Renda Fixa', 'Outros']:
        return 'Bom'
    elif perfil in ['Sofisticado', 'Arrojado'] and nivel_risco in ['Renda Variável', 'Outros']:
        return 'Bom'
    elif perfil in ['Sofisticado', 'Arrojado'] and nivel_risco == 'Renda Fixa':
        return 'Aceitável'
    elif perfil == 'Moderado' and nivel_risco == 'Renda Variável':
        return 'Aceitável'
    else:
        return 'Inadequado'

recomendacoes_com_perfil['alinhamento'] = recomendacoes_com_perfil.apply(avaliar_alinhamento, axis=1)
print("Qualidade do alinhamento das recomendações:")
alinhamento_counts = recomendacoes_com_perfil['alinhamento'].value_counts(normalize=True)
print(alinhamento_counts)



=== ALINHAMENTO PERFIL-RECOMENDAÇÃO ===
Qualidade do alinhamento das recomendações:
alinhamento
Bom          0.779492
Ótimo        0.142334
Aceitável    0.078174
Name: proportion, dtype: float64


## 4️⃣ Avaliação da Diversidade das Recomendações

Calcula quantos níveis de risco diferentes foram recomendados por conta e a média dessa diversidade.


In [82]:
print("\n=== ANÁLISE DE DIVERSIDADE ===")
diversidade_por_conta = recomendacoes_df.groupby('Conta')['Nivel de Risco'].nunique()
media_diversidade = diversidade_por_conta.mean()
print(f"Média de diferentes níveis de risco por conta: {media_diversidade:.2f}")


=== ANÁLISE DE DIVERSIDADE ===
Média de diferentes níveis de risco por conta: 1.39


## 5️⃣ Top 10 Ativos Mais Recomendados

Lista os ativos que mais apareceram nas recomendações.


In [83]:
print("\n=== ATIVOS MAIS RECOMENDADOS ===")
top_ativos = recomendacoes_df['Nome Ativo'].value_counts().head(10)
print(top_ativos)


=== ATIVOS MAIS RECOMENDADOS ===
Nome Ativo
LCI PRE        4067
CDCA CDIE      2460
LCI CDIE       2443
POTE12 IPCA    2258
MSGT23 IPCA    2153
RINV11         1615
DVLP11         1611
EQTL3          1565
VALEON         1562
GASP19 IPCA     353
Name: count, dtype: int64


## 6️⃣ Visualização da Distribuição das Recomendações

Cria gráficos para ilustrar:
- Qualidade de alinhamento
- Distribuição dos níveis de risco
- Recomendações por perfil
- Distribuição dos scores de recomendação

In [ ]:
plt.figure(figsize=(12, 8))

# Gráfico de barras para alinhamento
plt.subplot(2, 2, 1)
alinhamento_counts_abs = recomendacoes_com_perfil['alinhamento'].value_counts()
alinhamento_counts_abs.plot(kind='bar', color='skyblue')
plt.title('Qualidade do Alinhamento das Recomendações')
plt.xlabel('Classificação')
plt.ylabel('Quantidade')

# Gráfico de pizza para distribuição de níveis de risco
plt.subplot(2, 2, 2)
distribuicao_nivel_risco.plot(kind='pie', autopct='%1.1f%%')
plt.title('Distribuição por Nível de Risco')
plt.ylabel('')

# Distribuição de recomendações por perfil
plt.subplot(2, 2, 3)
crosstab = pd.crosstab(recomendacoes_com_perfil['perfil'], recomendacoes_com_perfil['Nivel de Risco'])
crosstab.plot(kind='bar', stacked=True)
plt.title('Distribuição de Recomendações por Perfil')
plt.xlabel('Perfil')
plt.ylabel('Quantidade')
plt.legend(title='Nível de Risco')

# Histograma de scores
plt.subplot(2, 2, 4)
plt.hist(recomendacoes_df['Score'], bins=20, color='green', alpha=0.7)
plt.title('Distribuição dos Scores de Recomendação')
plt.xlabel('Score')
plt.ylabel('Quantidade')

plt.tight_layout()
plt.savefig('analise_recomendacoes.png')
plt.show()

**Output da célula**

<div align="center">
<img src="img/output2.png" width="80%"><br>
</div>

<div align="center">
<img src="img/output1.png" width="80%"><br>
</div>

## 7️⃣ Cálculo de Métricas Específicas por Perfil

Para cada perfil de cliente:
- Calcula a quantidade de recomendações.
- Avalia a qualidade percentual ponderada do alinhamento.
- Mostra a distribuição de níveis de risco recomendados.


In [85]:
print("\n=== MÉTRICAS POR PERFIL DE CLIENTE ===")
for perfil in recomendacoes_com_perfil['perfil'].unique():
    recs_perfil = recomendacoes_com_perfil[recomendacoes_com_perfil['perfil'] == perfil]

    alinhamento_perfil = recs_perfil['alinhamento'].value_counts(normalize=True)
    qualidade_percentual = (
        (alinhamento_perfil.get('Ótimo', 0) * 1.0) +
        (alinhamento_perfil.get('Bom', 0) * 0.8) +
        (alinhamento_perfil.get('Aceitável', 0) * 0.5) +
        (alinhamento_perfil.get('Inadequado', 0) * 0.0)
    ) * 100

    print(f"\nPerfil: {perfil}")
    print(f"Quantidade de recomendações: {len(recs_perfil)}")
    print(f"Qualidade de alinhamento: {qualidade_percentual:.2f}%")
    print(f"Distribuição por tipo: {recs_perfil['Nivel de Risco'].value_counts(normalize=True)}")



=== MÉTRICAS POR PERFIL DE CLIENTE ===

Perfil: Moderado
Quantidade de recomendações: 9480
Qualidade de alinhamento: 80.00%
Distribuição por tipo: Nivel de Risco
Renda Fixa    1.0
Name: proportion, dtype: float64

Perfil: Sofisticado
Quantidade de recomendações: 8085
Qualidade de alinhamento: 74.06%
Distribuição por tipo: Nivel de Risco
Renda Variável    0.801979
Renda Fixa        0.198021
Name: proportion, dtype: float64

Perfil: Conservador
Quantidade de recomendações: 2915
Qualidade de alinhamento: 100.00%
Distribuição por tipo: Nivel de Risco
Renda Fixa    1.0
Name: proportion, dtype: float64


## 8️⃣ Avaliação de desempenho do modelo com métricas de erro

Nesta célula, avaliamos o desempenho do modelo `SVD` treinado utilizando as métricas **RMSE (Root Mean Squared Error)** e **MAE (Mean Absolute Error)**. Para isso, foi criado um `testset` a partir dos próprios dados de treino (`trainset.build_testset()`), e as previsões foram realizadas usando o método `model.test(testset)`.

As métricas foram então calculadas com:

- `accuracy.rmse(predictions)` → erro quadrático médio.
- `accuracy.mae(predictions)` → erro absoluto médio.

Essa avaliação permite ter uma noção da precisão do modelo sobre os dados que ele viu durante o treinamento.



In [86]:
from surprise import accuracy

testset = trainset.build_testset()
predictions = model.test(testset)

accuracy.rmse(predictions)
accuracy.mae(predictions)

RMSE: 0.1674
MAE:  0.1134


0.11342488313940208

## 9️⃣ Análise dos Resultados

* **RMSE: 0.1668**
* **MAE: 0.1128**

Os resultados indicam que, em média, a diferença entre as previsões do modelo e os valores reais está entre **0.11 e 0.16** em uma escala de rating de **0 a 1**.

Vale destacar que essa avaliação foi realizada utilizando o próprio conjunto de treinamento (via `build_testset()` no Surprise), o que naturalmente tende a apresentar erros mais baixos, já que o modelo está avaliando dados que já conhece. Ainda assim, os valores obtidos são consistentes e indicam que o modelo é capaz de reproduzir razoavelmente bem os padrões aprendidos no conjunto de dados utilizado.

Para uma avaliação mais robusta e realista, vamos futuramente validar o desempenho do modelo em um conjunto de dados separado (holdout ou cross-validation) para verificar sua capacidade de generalização.

---


## Conclusão:

**Modelo escolhido:** Abordagem *Model-Based* utilizando o algoritmo **SVD** da biblioteca `surprise`. 

Foram testadas outras abordagens, incluindo o **KNN** com métrica **Pearson Baseline**, mas os resultados não foram satisfatórios, devido à alta concentração de clientes com proporção **1.0 de renda fixa**, os resultados acabaram sendo muito homogêneos e pouco representativos do comportamento real dos usuários. Isso impactou negativamente a capacidade dos modelos de capturar variações relevantes no perfil dos clientes.

O resultado dessa abordagem alternativa pode ser conferido na célula abaixo:

---

In [ ]:
testeKNN = pd.read_csv("recomendacoes_por_contaKNN.csv")

testeKNN.head(25)

,Conta,Nome Ativo,Score,Nivel de Risco,Impacto
0,27,GSTS24 IPCA,1.200000,Renda Fixa,Recomendado para rebalanceamento: aumenta Rend...
1,27,CRTR12 IPCA,1.200000,Renda Fixa,Recomendado para rebalanceamento: aumenta Rend...
2,27,CBAN52 IPCA,1.200000,Renda Fixa,Recomendado para rebalanceamento: aumenta Rend...
3,27,VERO13 IPCA,1.200000,Renda Fixa,Recomendado para rebalanceamento: aumenta Rend...
4,27,ASER12 IPCA,1.200000,Renda Fixa,Recomendado para rebalanceamento: aumenta Rend...
5,44,RAIZ4,1.200000,Renda Variável,Recomendado para rebalanceamento: aumenta Rend...
6,44,AMOB3,1.200000,Renda Variável,Recomendado para rebalanceamento: aumenta Rend...
7,44,ABEV3,1.200000,Renda Variável,Recomendado para rebalanceamento: aumenta Rend...
8,44,AZZA3,1.200000,Renda Variável,Recomendado para rebalanceamento: aumenta Rend...
9,44,LREN3,1.200000,Renda Variável,Recomendado para rebalanceamento: aumenta Rend...
